# Notebook 24: teaching the model what a weak compound looks like (CYP2D6 and CYP1A2)

**The problem.** The organisers confirmed in the challenge Discord (2026-10-08) that a blank
pIC50 in the training data means the compound was not a primary-screen hit for that isoform,
so no dose-response was run, and that the test set was built densely, with every compound
measured on every isoform. On CYP2D6 the cut is clean in our data: all 1,493 labelled
compounds respond at -0.59 or stronger in the single-concentration screen, and of the 2,882
blank compounds that have screen data only 5 respond at -0.73 or stronger. Every model we have
trained on CYP2D6 has seen only compounds on the strong side of that cut, while the test set
also contains the weak ones.

**The change.** Training rows for a blank cell, `(compound, isoform)`, are added with a low
target (`FLOOR`, default 3.5) when the screen says the compound is clearly weak on that
isoform: blank, a measured screen response, and a response weaker than the 95th percentile of
the labelled compounds' responses (-0.72 for CYP2D6, -0.18 for CYP1A2: 2,877 and 1,409 cells).
The real screen response only *defines* these rows. It is not a model input, so the model
never sees a feature that the blind compounds lack. CYP2C9 and CYP3A4 are left out because
their blanks include strong responders (CYP3A4 blanks respond more strongly than its labelled
compounds), so blank does not mean weak there.

**The model** is the nb15 recipe, a TabPFN regressor on 200 components of the
`surrogate_adme` embedding beside the five qHTS and four aux columns kept raw, so nothing else
differs between the two variants:

| variant | training rows |
| :--- | :--- |
| `base` | labelled compounds only (the control) |
| `pseudo` | labelled compounds plus pseudo-labelled blank cells, at most `PSEUDO_RATIO` (1.0) times as many as the labelled rows |

**The check the sparse holdout cannot give.** Spearman on labelled compounds can only fall or
stay level here, because the labelled compounds are exactly the ones the base model already
fits. So each variant is also scored by **AUC of labelled versus clean-blank cells on
held-out compounds**: for compounds held out of training, how often is the prediction for a
labelled cell higher than for a clean-blank cell. Computed on the five development folds and
on the holdout. A model that has learned what a weak compound looks like should separate them.

**Gate.** Worth a blend test if, on an isoform, `pseudo` raises AUC by at least 0.03 on dev
and on holdout while labelled-cell Spearman falls by no more than 0.01. The blend test swaps
that isoform's `n15` column for the `pseudo` arm in the `v25` pipeline (the arm needs its own
calibration, since predictions for weak compounds move down). The board is the only real test
of the dense population.

**Unverified assumptions.** Blank means below the assay limit and not merely unmeasured; the
floor value; and that a pseudo-label at 3.5 teaches weakness without teaching a cliff. If the
floor is wrong, AUC rises while Spearman and the board get worse, which is why both are shown.

**Reference**, nb15 (all four isoforms, `surrogate_adme`): dev Spearman 0.5639 / 0.7268 /
**0.4460** / 0.8115, holdout macro 0.6713. The `base` variant here should land close to nb15's
CYP1A2 and CYP2D6 values; it will not match exactly because the folds are drawn over all
development compounds rather than over each isoform's labelled ones.

**Time:** not measured. Four runs (two isoforms, two variants) of seven TabPFN fits, the
`pseudo` ones on roughly twice as many rows. Budget one to two hours on a T4.

**Safe to run beside notebooks 19 to 25, and beside copies of itself with different settings.**
It reads the nb12 embedding of `surrogate_adme` and the shared payload, and writes only `nb24_*`
files (`nb24_<RUN>_<variant>_<iso>.npz`, `part_nb24_<RUN>_*`, `nb24_pca_<encoder>.npz`,
`results_nb24_<RUN>.json`), each saved under a random temporary name and renamed. **Give every
runtime its own `RUN` letter**, because the letter is in the file names; two runtimes with the same
letter and different settings would overwrite each other. The zip holds only this run's files.
Run all resumes after a disconnect.

**Settings run so far and planned** (CYP2D6 only for the new ones; the `base` control is already
done as run `a`, so set `VARIANTS = ['pseudo']` and `ISO_RUN = ['CYP2D6']`):

| `RUN` | `FLOOR` | `PSEUDO_RATIO` | note |
| :--- | ---: | ---: | :--- |
| `a` | 3.5 | 1.0 | done; `v26` on the board used this arm (rank 55 to 43) |
| `b` | 3.5 | 2.0 | more weak rows per labelled row |
| `c` | 3.0 | 1.0 | lower floor |
| `d` | 4.0 | 1.0 | higher floor |
| `e` | 3.5 | 1.0 | `ENCODER = 'medium'`: the same idea on the `chemprop_medium` embedding, a second, differently built arm |

Each prints, besides AUC and Spearman, the mean, spread and share below 3.8 of its predictions
for the 750 blind compounds. The sparse holdout cannot rank these settings by their effect on the
dense test set; the AUC and the blind-set shift are the guides, and the board is the judge.

In [23]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to a GPU before going on')

Fri Oct 23 21:19:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   74C    P0             33W /   72W |   21986MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
RUN = 'e'                       # one letter per runtime AND per FLOOR / PSEUDO_RATIO setting: it names the output files
ISO_RUN = ['CYP2D6', 'CYP1A2']  # the two isoforms whose blanks are screen-selected
VARIANTS = ['base', 'pseudo']
FLOOR = 3.5                    # pIC50 given to a clean-blank cell
PSEUDO_RATIO = 1.0              # at most this many pseudo rows per labelled row, per fit
ENCODER = 'medium'      # nb12 embedding on Drive (nb15's encoder), or 'medium' for chemprop_medium from the payload
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
assert os.path.ismount('/content/drive'), 'Drive is not mounted; the run could not resume without it'

NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy', 'scaffold_groups.npy',
        'qhts_tr.npy', 'qhts_te.npy', 'aux_train.npy', 'aux_test.npy']
if ENCODER == 'medium': NEED += ['emb_medium_inhib.npy', 'emb_medium_test.npy']
have = sorted(os.listdir('/content/cache'))
missing = [f for f in NEED if f not in have]
assert not missing, 'payload is missing %s. Found: %s' % (missing, have)
gone = [] if ENCODER == 'medium' else [f for f in (f'fnd_{ENCODER}_train.npy', f'fnd_{ENCODER}_test.npy')
        if not os.path.exists(f'{OUTDIR}/{f}')]
assert not gone, f'nb12 embedding missing from {OUTDIR}: {gone}. Rerun the embedding step of cyp_nb12_foundation.'
print('run', RUN, '| payload ok | embedding found |', ISO_RUN, VARIANTS, '| floor', FLOOR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
run e | payload ok | embedding found | ['CYP2D6', 'CYP1A2'] ['base', 'pseudo'] | floor 3.5


/tmp/ipykernel_852/2693139473.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall('/content/cache')


In [25]:
!pip -q install tabpfn==2.2.1

In [26]:
import json, os, time, uuid, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import rankdata, spearmanr
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISO = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
meta = pd.read_csv(CACHE / 'meta_train.csv.gz'); mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy'); G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
RAW = np.hstack([np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'aux_train.npy')]).astype(np.float32)
RAW_TE = np.hstack([np.load(CACHE / 'qhts_te.npy'), np.load(CACHE / 'aux_test.npy')]).astype(np.float32)
N, NTE = len(meta), len(mete)
assert len(RAW) == N == len(hold) == len(G) and len(RAW_TE) == NTE
Y = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
SCREEN = {i: meta[f'{i}_log2fc'].to_numpy(float) for i in ISO}   # training compounds only

def save_atomic(path, **kw):
    """Own temporary name, then rename: a half-written file is never read."""
    tmp = Path(f'{path}.{uuid.uuid4().hex}.tmp.npz')
    np.savez_compressed(tmp, **kw); os.replace(tmp, path)

def pca_features(name, k=200):
    """200 label-free components of one encoder's frozen embedding, fitted on train and
    test molecules together as the nb15 payload was. Cached on Drive under this notebook's name."""
    f = OUT / f'nb24_pca_{name}.npz'
    if f.exists():
        z = np.load(f); return z['tr'], z['te']
    if name == 'medium':
        a = np.load(CACHE / 'emb_medium_inhib.npy').astype(np.float32); b = np.load(CACHE / 'emb_medium_test.npy').astype(np.float32)
    else:
        a = np.load(OUT / f'fnd_{name}_train.npy').astype(np.float32)
        b = np.load(OUT / f'fnd_{name}_test.npy').astype(np.float32)
    assert a.shape[0] == N and b.shape[0] == NTE and a.shape[1] == b.shape[1], (name, a.shape, b.shape)
    X = np.vstack([a, b]); X[~np.isfinite(X)] = np.nan
    X = SimpleImputer(strategy='median').fit_transform(X)
    X = X[:, X.std(0) > 1e-8]
    X = StandardScaler().fit_transform(X)
    P = PCA(n_components=min(k, X.shape[1]), random_state=42).fit_transform(X).astype(np.float32)
    save_atomic(f, tr=P[:N], te=P[N:])
    return P[:N], P[N:]

PF, PF_TE = pca_features(ENCODER)
print('components', PF.shape, PF_TE.shape)

components (4905, 200) (750, 200)


## Which blank cells count as clearly weak

A cell is a clean blank when the compound has no pIC50 for the isoform, has a measured screen
response for it, and that response is weaker than the 95th percentile of the labelled
development compounds' responses. The cut uses development compounds only.

In [27]:
obs_any = np.isfinite(np.column_stack([Y[i] for i in ISO])).any(1)
dev_all = np.flatnonzero(obs_any & ~hold); ho_all = np.flatnonzero(obs_any & hold)
CLEAN, CUT = {}, {}
for iso in ISO_RUN:
    lab = np.isfinite(Y[iso]); f = SCREEN[iso]
    ref = f[lab & ~hold & np.isfinite(f)]
    CUT[iso] = float(np.percentile(ref, 95))
    CLEAN[iso] = (~lab) & np.isfinite(f) & (f > CUT[iso])
    lo, hi = float(ref.min()), float(ref.max())
    print(f'{iso}: labelled dev screen response {lo:.2f} to {hi:.2f} (95th pct {CUT[iso]:.2f}) | '
          f'clean blank cells {int(CLEAN[iso].sum())} '
          f'(dev compounds {int((CLEAN[iso] & ~hold).sum())}, holdout {int((CLEAN[iso] & hold).sum())}) | '
          f'labelled cells {int(lab.sum())}')
    lowest = np.sort(Y[iso][lab & ~hold])[:int(0.1 * (lab & ~hold).sum())]
    print(f'   floor {FLOOR} sits at the {100*np.mean(Y[iso][lab & ~hold] <= FLOOR):.0f}th percentile '
          f'of the labelled dev pIC50 values')
FOLDS = list(GroupKFold(5).split(dev_all, dev_all, G[dev_all]))
fold_of = np.full(N, -1)
for k, (tr, va) in enumerate(FOLDS):
    fold_of[dev_all[va]] = k
print('development compounds', len(dev_all), '| holdout compounds', len(ho_all), '| folds',
      [len(va) for _, va in FOLDS])

CYP2D6: labelled dev screen response -3.27 to -0.59 (95th pct -0.72) | clean blank cells 2877 (dev compounds 2452, holdout 425) | labelled cells 1493
   floor 3.5 sits at the 7th percentile of the labelled dev pIC50 values
CYP1A2: labelled dev screen response -2.90 to 2.70 (95th pct -0.18) | clean blank cells 1409 (dev compounds 1188, holdout 221) | labelled cells 1412
   floor 3.5 sits at the 10th percentile of the labelled dev pIC50 values
development compounds 4179 | holdout compounds 726 | folds [836, 836, 836, 836, 835]


## Fit, predict, and score

For each isoform and variant: five folds over all development compounds (so blank compounds
are held out too), a holdout fit on all development compounds, and a blind-set fit on
development plus holdout. Pseudo rows only ever come from the compounds the fit trains on. Every fold
is saved the moment it finishes.

In [ ]:
from tabpfn import TabPFNRegressor
KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)

def train_rows(iso, comps, variant, seed):
    lab = np.isfinite(Y[iso]); L = comps[lab[comps]]
    if variant == 'base':
        return L, Y[iso][L]
    P = comps[CLEAN[iso][comps]]
    take = min(len(P), int(PSEUDO_RATIO * len(L)))
    P = np.random.default_rng(seed).choice(P, take, replace=False)
    return np.r_[L, P], np.r_[Y[iso][L], np.full(len(P), FLOOR)]

def fit_predict(rows, yvec, ev_P, ev_RAW):
    ri = SimpleImputer(strategy='median').fit(RAW[rows])
    Xt = np.hstack([PF[rows], ri.transform(RAW[rows])]).astype(np.float32)
    Xe = np.hstack([ev_P, ri.transform(ev_RAW)]).astype(np.float32)
    return TabPFNRegressor(**KW).fit(Xt, yvec).predict(Xe)

def auc(p, pos, neg):
    """P(prediction for a labelled cell > prediction for a clean-blank cell)."""
    if len(pos) == 0 or len(neg) == 0: return float('nan')
    r = rankdata(np.r_[p[pos], p[neg]]); n1, n0 = len(pos), len(neg)
    return float((r[:n1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

RESULTS = []; t_all = time.time()
for iso in ISO_RUN:
    lab = np.isfinite(Y[iso]); clean = CLEAN[iso]
    for variant in VARIANTS:
        final = OUT / f'nb24_{RUN}_{variant}_{iso}.npz'
        part = OUT / f'part_nb24_{RUN}_{variant}_{iso}.npz'
        dv = np.flatnonzero(lab & ~hold); ho = np.flatnonzero(lab & hold)
        oof_f = np.full(N, np.nan); hp_f = np.full(N, np.nan); tp = np.full(NTE, np.nan); done = []
        if final.exists():
            z = np.load(final, allow_pickle=True)
            oof_f, hp_f, tp = z['oof_full'], z['hp_full'], z['test']; done = ['all']
            print(f'{iso} {variant}: finished earlier, skipping', flush=True)
        else:
            if part.exists():
                z = np.load(part, allow_pickle=True)
                oof_f, hp_f, tp, done = z['oof_f'], z['hp_f'], z['tp'], list(z['done'])
                print(f'{iso} {variant}: resuming, finished stages {done}', flush=True)
            def ckpt(): save_atomic(part, oof_f=oof_f, hp_f=hp_f, tp=tp, done=np.array(done))
            t0 = time.time()
            for k, (tr, va) in enumerate(FOLDS):
                if f'f{k}' in done: continue
                rows, yv = train_rows(iso, dev_all[tr], variant, 42 + k)
                ev = dev_all[va]
                oof_f[ev] = fit_predict(rows, yv, PF[ev], RAW[ev])
                done.append(f'f{k}'); ckpt()
                print(f'  {iso} {variant} fold {k}: {len(rows)} training rows ({time.time()-t0:.0f}s)', flush=True)
            if 'hold' not in done:
                rows, yv = train_rows(iso, dev_all, variant, 142)
                hp_f[ho_all] = fit_predict(rows, yv, PF[ho_all], RAW[ho_all]); done.append('hold'); ckpt()
                print(f'  {iso} {variant} holdout model ({time.time()-t0:.0f}s)', flush=True)
            if 'test' not in done:
                rows, yv = train_rows(iso, np.r_[dev_all, ho_all], variant, 242)
                tp = fit_predict(rows, yv, PF_TE, RAW_TE); done.append('test'); ckpt()
                print(f'  {iso} {variant} test model ({time.time()-t0:.0f}s)', flush=True)
            assert np.isfinite(oof_f[dev_all]).all() and np.isfinite(hp_f[ho_all]).all() and np.isfinite(tp).all()
            save_atomic(final, oof=oof_f[dv], dev=dv, hold=hp_f[ho], ho=ho, test=tp,
                        oof_full=oof_f, hp_full=hp_f)
            part.unlink(missing_ok=True)
        a_dev = [auc(oof_f, np.flatnonzero(lab & (fold_of == k)), np.flatnonzero(clean & (fold_of == k)))
                 for k in range(5)]
        a_ho = auc(hp_f, np.flatnonzero(lab & hold), np.flatnonzero(clean & hold))
        d = float(spearmanr(Y[iso][dv], oof_f[dv]).statistic); h = float(spearmanr(Y[iso][ho], hp_f[ho]).statistic)
        r = dict(iso=iso, variant=variant, dev_spearman=d, holdout_spearman=h,
                 auc_dev=float(np.nanmean(a_dev)), auc_dev_folds=a_dev, auc_holdout=a_ho,
                 mean_pred_labelled=float(np.nanmean(oof_f[lab & ~hold])),
                 mean_pred_clean_blank=float(np.nanmean(oof_f[clean & ~hold])),
                 blind_mean=float(tp.mean()), blind_sd=float(tp.std()), blind_below_3p8=float((tp < 3.8).mean()),
                 floor=FLOOR, pseudo_ratio=PSEUDO_RATIO)
        RESULTS.append(r); json.dump(RESULTS, open(OUT / f'results_nb24_{RUN}.json', 'w'), indent=2)
        print(f'{iso} {variant}: dev rho {d:.4f} | holdout rho {h:.4f} | AUC dev {r["auc_dev"]:.3f} '
              f'(folds {" ".join(f"{x:.2f}" for x in a_dev)}) | AUC holdout {a_ho:.3f} | '
              f'mean prediction: labelled {r["mean_pred_labelled"]:.2f}, clean blank {r["mean_pred_clean_blank"]:.2f} | '
              f'blind set: mean {r["blind_mean"]:.2f} sd {r["blind_sd"]:.2f}, {100*r["blind_below_3p8"]:.0f}% below 3.8',
              flush=True)

R = pd.DataFrame(RESULTS)
print('\n' + R[['iso', 'variant', 'dev_spearman', 'holdout_spearman', 'auc_dev', 'auc_holdout']].round(4).to_string(index=False))
for iso in ISO_RUN:
    b = R[(R.iso == iso) & (R.variant == 'base')]; p = R[(R.iso == iso) & (R.variant == 'pseudo')]
    if len(b) and len(p):
        b, p = b.iloc[0], p.iloc[0]
        print(f'\n{iso}: AUC dev {p.auc_dev-b.auc_dev:+.3f}, holdout {p.auc_holdout-b.auc_holdout:+.3f} | '
              f'labelled Spearman dev {p.dev_spearman-b.dev_spearman:+.4f}, holdout {p.holdout_spearman-b.holdout_spearman:+.4f}')
        ok = (p.auc_dev - b.auc_dev >= 0.03 and p.auc_holdout - b.auc_holdout >= 0.03
              and p.dev_spearman - b.dev_spearman >= -0.01 and p.holdout_spearman - b.holdout_spearman >= -0.01)
        print('   gate (AUC +0.03 on dev and holdout, Spearman down at most 0.01):', 'PASS' if ok else 'FAIL')
print('nb15 for reference: dev Spearman CYP1A2 0.5639, CYP2D6 0.4460')
print('total', round((time.time() - t_all) / 60, 1), 'min')

  CYP2D6 base fold 0: 1004 training rows (12s)
  CYP2D6 base fold 1: 1017 training rows (22s)
  CYP2D6 base fold 2: 1015 training rows (32s)
  CYP2D6 base fold 3: 1043 training rows (43s)
  CYP2D6 base fold 4: 1013 training rows (53s)
  CYP2D6 base holdout model (65s)
  CYP2D6 base test model (77s)
CYP2D6 base: dev rho 0.4209 | holdout rho 0.5262 | AUC dev 0.757 (folds 0.75 0.73 0.77 0.77 0.76) | AUC holdout 0.743 | mean prediction: labelled 4.83, clean blank 4.47 | blind set: mean 4.58 sd 0.47, 2% below 3.8
  CYP2D6 pseudo fold 0: 2008 training rows (15s)
  CYP2D6 pseudo fold 1: 2034 training rows (32s)
  CYP2D6 pseudo fold 2: 2030 training rows (47s)
  CYP2D6 pseudo fold 3: 2086 training rows (63s)
  CYP2D6 pseudo fold 4: 2026 training rows (79s)
  CYP2D6 pseudo holdout model (97s)
  CYP2D6 pseudo test model (121s)
CYP2D6 pseudo: dev rho 0.4449 | holdout rho 0.5140 | AUC dev 0.826 (folds 0.81 0.85 0.83 0.83 0.81) | AUC holdout 0.821 | mean prediction: labelled 4.42, clean blank 3.82 

In [ ]:
# Only this run's own files, never the whole folder, which other notebooks may be writing to.
import zipfile
mine = ([OUT / f'nb24_{RUN}_{v}_{i}.npz' for v in VARIANTS for i in ISO_RUN] + [OUT / f'results_nb24_{RUN}.json'])
with zipfile.ZipFile(f'/content/nb24_{RUN}_results.zip', 'w') as z:
    for p in mine:
        z.write(p, p.name)
try:
    from google.colab import files
    files.download(f'/content/nb24_{RUN}_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- the results are already on Drive in', OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>